In [20]:
import os
import pandas as pd
import re

# ----------------------------
# SETTINGS
# ----------------------------
WIKI_TREATIES = "./wiki-treaties_formatted.csv"
UNO_TREATIES = "./UNO-Treaties.csv"
INSTRUMENTS = "../instruments/ohchr_instruments_detailed.csv"
CONV_PROT_REC = "../conv-prot-rec/conventions-protocols-recommendations.csv"
RESOLUTIONS_TRIM = "../resolutions/ga_resolutions_1946_2019_3100-trim.csv"

df_wiki = pd.read_csv(WIKI_TREATIES)
df_uno = pd.read_csv(UNO_TREATIES)
df_ins = pd.read_csv(INSTRUMENTS)
df_conv = pd.read_csv(CONV_PROT_REC)
df_res = pd.read_csv(RESOLUTIONS_TRIM)

df_wiki.head()

,year,name,url,note,summary,cleaned_note,cleaned_title
0,1900,Treaty of Paris,https://en.wikipedia.org/wiki/Treaty_of_Paris_...,NaN,Ends all conflicting claims over Río Muni ( Eq...,NaN,Treaty of Paris
1,1900,Treaty of Washington,https://en.wikipedia.org/wiki/Treaty_of_Washin...,NaN,Seeks to remove any ground of misunderstanding...,NaN,Treaty of Washington
2,1900,Convention for the Preservation of Wild Animal...,https://en.wikipedia.org/wiki/Convention_for_t...,NaN,First international agreement on wildlife cons...,NaN,Convention for the Preservation of Wild Animal...
3,1901,Hay–Pauncefote Treaty,https://en.wikipedia.org/wiki/Hay%E2%80%93Paun...,NaN,Replaces the Clayton–Bulwer Treaty,NaN,Hay–Pauncefote Treaty
4,1901,Boxer Protocol,https://en.wikipedia.org/wiki/Boxer_Protocol,Also known as the Peace Agreement between the ...,Peace agreement between the Eight-Nation Allia...,Peace Agreement between the Great Powers and C...,Boxer Protocol


In [21]:
print(f"WIKI_TREATIES columns: {df_wiki.columns}")
print(f"UNO_TREATIES columns: {df_uno.columns}")
print(f"INSTRUMENTS columns: {df_ins.columns}")
print(f"CONV_PROT_REC columns: {df_conv.columns}")
print(f"RESOLUTIONS_TRIM columns: {df_res.columns}")

WIKI_TREATIES columns: Index(['year', 'name', 'url', 'note', 'summary', 'cleaned_note',
       'cleaned_title'],
      dtype='object')
UNO_TREATIES columns: Index(['title', 'location', 'date', 'chapter'], dtype='object')
INSTRUMENTS columns: Index(['title', 'url', 'adopted_by', 'content', 'pdf_url'], dtype='object')
CONV_PROT_REC columns: Index(['code', 'title', 'year', 'number'], dtype='object')
RESOLUTIONS_TRIM columns: Index(['res_id2', 'part', 'res_id3', 'res_id2_unlet', 'alt_id_dic',
       'session_type', 'session_reg', 'session_sp', 'session_es', 'resn',
       'res_letter', 'date_p', 'date_c', 'filename', 'content', 'location',
       'record', 'draft', 'topic', 'n_inc_cit'],
      dtype='object')


In [22]:
print(f"{'WIKI_TREATIES':=^30}")
print(f"Number of unique cleaned titles: {df_wiki['cleaned_title'].nunique()}\n\
        Length of all titles: {len(df_wiki['cleaned_title'])}")
print(f"Number of unique cleaned alternative names: {df_wiki['cleaned_note'].nunique()}\n\
        Length of all alternative names: {len(df_wiki['cleaned_note'].dropna())}")


print("Non-unique cleaned titles:")
non_unique_titles = df_wiki[df_wiki.duplicated(subset='cleaned_title', keep=False)]
non_unique_titles[['year', 'cleaned_title', 'name']]

========WIKI_TREATIES=========
Number of unique cleaned titles: 345
        Length of all titles: 364
Number of unique cleaned alternative names: 101
        Length of all alternative names: 102
Non-unique cleaned titles:


,year,cleaned_title,name
21,1905,Japan–Korea Treaty,Japan–Korea Treaty of 1905
28,1910,Japan–Korea Treaty,Japan–Korea Treaty of 1910
34,1913,Treaty of London,Treaty of London (1913)
35,1913,Treaty of Bucharest,Treaty of Bucharest (1913)
40,1915,Treaty of London,Treaty of London (1915) (London Pact)
44,1916,Treaty of Bucharest,Treaty of Bucharest (1916)
51,1918,Treaty of Bucharest,Treaty of Bucharest (1918)
64,1920,Treaty of Warsaw,Treaty of Warsaw (1920)
66,1920,Treaty of Rapallo,Treaty of Rapallo (1920)
67,1920,Treaty of Moscow,Treaty of Moscow (1920)


In [23]:
print(f"{'UNO_TREATIES':=^30}")
print(f"Number of unique titles: {df_uno['title'].nunique()}\n\
        Length of all titles: {len(df_uno['title'])}")

dupes = (
    df_uno.groupby(["title", "date"])
          .size()
          .reset_index(name="count")
          .query("count > 1")
)
print('Duplicates considering the tile and the date:')
print(dupes)

=========UNO_TREATIES=========
Number of unique titles: 399
        Length of all titles: 427
Duplicates considering the tile and the date:
Empty DataFrame
Columns: [title, date, count]
Index: []


In [24]:
'''
De los titulos que si son unicos, podemos usarlos directamente, sin especificar el año.
De los titulos que no son unicos, tenemos que usar el año. Podemos usarlo de las siguientes maneras
- El titulo seguido de "of YEAR"
- El titulo seguido de (YEAR)
- El titulo precedido por YEAR
- El titulo precedido por (YEAR)
Solo el titulo, extraer el fragmento con un padding de 10 caracteres y decidir
'''

'\nDe los titulos que si son unicos, podemos usarlos directamente, sin especificar el año.\nDe los titulos que no son unicos, tenemos que usar el año. Podemos usarlo de las siguientes maneras\n- El titulo seguido de "of YEAR"\n- El titulo seguido de (YEAR)\n- El titulo precedido por YEAR\n- El titulo precedido por (YEAR)\nSolo el titulo, extraer el fragmento con un padding de 10 caracteres y decidir\n'

In [25]:
"""
1. create a df with the unique titles (cleaned_name column)
2. scan them through the resolution['content]
3. save all in a cites dataframe
4. now the non unique titles (cleaned_name column)
5. scan them non alone through the resolution['content]: they must be accompagned with 'of df['year'], ' (df['year'])', 'df['year'] df['cleaned_name']', '(df['year']) df['cleaned_name']'
6. save everything in cites dataframe
Note: the cites dataframe is resolutions[res_id2], treaty and id, which is composed of the treaty + comma + year
"""


"\n1. create a df with the unique titles (cleaned_name column)\n2. scan them through the resolution['content]\n3. save all in a cites dataframe\n4. now the non unique titles (cleaned_name column)\n5. scan them non alone through the resolution['content]: they must be accompagned with 'of df['year'], ' (df['year'])', 'df['year'] df['cleaned_name']', '(df['year']) df['cleaned_name']'\n6. save everything in cites dataframe\nNote: the cites dataframe is resolutions[res_id2], treaty and id, which is composed of the treaty + comma + year\n"

In [28]:
# Lowercase, otherwise nothing is detected
df_wiki["cleaned_title"] = df_wiki["cleaned_title"].str.lower()

# -------------------------
# Separate unique/non-unique
# -------------------------
unique_titles = df_wiki.drop_duplicates("cleaned_title", keep=False)

non_unique_titles = df_wiki[
    df_wiki.duplicated("cleaned_title", keep=False)
]

# -------------------------
# Collect matches
# -------------------------
records = []

# =====================================================
# 1. UNIQUE TITLES
# =====================================================
print("Scanning unique titles...")
for _, treaty in unique_titles.iterrows():

    title = treaty["cleaned_title"]
    year = treaty["year"]

    treaty_id = f"{title};{year}"

    pattern = re.escape(title)

    for _, res in df_res.iterrows():

        content = res["content"]

        if title not in content:
            continue

        # Only now run regex
        if re.search(pattern, content):
            records.append({
                "res_id2": res["res_id2"],
                "treaty": title,
                "id": treaty_id
            })


# =====================================================
# 2. NON-UNIQUE TITLES (Optimized)
# =====================================================

MAX_WORD_DIST = 5
print("Scanning non-unique titles...")
for _, treaty in non_unique_titles.iterrows():

    title = treaty["cleaned_title"]
    year = str(int(treaty["year"]))
    treaty_id = f"{title};{year}"

    # Precompile patterns once per treaty
    patterns = [
        rf"\b{re.escape(title)}\b(?:\W+\w+){{0,{MAX_WORD_DIST}}}\W+{year}\b",
        rf"\b{year}\b(?:\W+\w+){{0,{MAX_WORD_DIST}}}\W+{re.escape(title)}\b",
        rf"\b{re.escape(title)}\s*\({year}\)",
        rf"\({year}\)\s*{re.escape(title)}",
        rf"\b{re.escape(title)}\s+of\s+{year}\b",
    ]

    for _, res in df_res.iterrows():

        content = res["content"]

        if title not in content:
            continue

        # Only now run expensive regex
        if any(re.search(p, content) for p in patterns):
            records.append({
                "res_id2": res["res_id2"],
                "treaty": title,
                "id": treaty_id
            })


# -------------------------
# Final dataframe
# -------------------------
cites = (
    pd.DataFrame(records)
      .drop_duplicates()
      .reset_index(drop=True)
)

print(cites.head())
print(f"\nFound {len(cites)} treaty citations.")

Scanning unique titles...
Scanning non-unique titles...
     res_id2                                             treaty  \
0     61 (i)                                     rome agreement   
1  2021 (xx)  convention and statute on the international ré...   
2  2021 (xx)  international convention for the suppression o...   
3     62/215                                    london protocol   
4     63/111                                    london protocol   

                                                  id  
0                                rome agreement;1907  
1  convention and statute on the international ré...  
2  international convention for the suppression o...  
3                               london protocol;1944  
4                               london protocol;1944  

Found 2876 treaty citations.


In [29]:
cites.to_csv("treaty_citations-first-draft.csv", index=False)

In [ ]:
"""


Geneva convention. there are many, the first is just geneva convention but the other ones are named by second, third, forth geneva convention. The problem is that the cases where the forth is references, sometimes they dont inlcude the ordinal, so the system detects as the first, but the text next inlcude the date, including 1949
Solution, the geneva convention should also detects a date
Geneva Convention relative to the Protection of Civilian Persons in Time of War, of 12 August 1949 = fourth geneva convention


"""

In [ ]:
# group to see the most cited treaties
cites = (
    cites.groupby('id')
         .agg(count=('res_id2', 'count'))
         .reset_index()
)

In [ ]:
cites.sort_values(by='count', ascending=False, inplace=True)
print(cites.head(10))